# 26 · Evaluate tool use and factual output

Evaluate business facts and tool trajectories, not whether prose is identical. A tiny live case below uses Python assertions without an LLM judge.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** About 2 Lite calls; native eval is explicit opt-in. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
import json
agent = llm("eval_shop", "Use get_order. Include the order ID and exact status in the response.", tools=[get_order])
lab = await make_lab(agent)
events = await ask(lab, "What is the status of O1001?")
calls = [c for e in events for c in e.get_function_calls()]
answer = final_text(events)
checks = {
    "correct_tool": any(c.name == "get_order" and c.args.get("order_id") == "O1001" for c in calls),
    "correct_order": "O1001" in answer,
    "correct_status": "delivered" in answer.lower(),
}
print(checks)
assert all(checks.values())


In [ ]:
# A native ADK EvalSet is included for adk eval / AgentEvaluator.
from google.adk.evaluation.eval_set import EvalSet
eval_set = EvalSet.model_validate_json((ROOT / "evals" / "orders.evalset.json").read_text())
print("Native evaluation cases:", len(eval_set.eval_cases))


From this directory, `adk eval shop_app evals/orders.evalset.json --config_file_path evals/eval_config.json` runs the native evaluator. It makes model calls. The config uses tool trajectory matching only, avoiding an LLM judge. ADK also offers response, safety, grounding and multi-turn simulation metrics; those can add model calls and optional dependencies.


## Try it

Add O1002 and an unknown order as regression cases. Keep evaluation data separate from instructions.


## Reference

[Official ADK documentation](https://adk.dev/evaluate/). Shared configuration: `config.json`. No environment activation or login cells are needed.
